# Spike: interrupt inside a parallel superstep

The two questions the translator's design is blocked on:

- **(a) Piercing** — while one parallel branch is paused on `interrupt()`, do the *sibling* branches' updates surface on the stream, or does the runtime hold everything until the superstep commits?
  - siblings surface **before** the interrupt → the wire sees mid-superstep partials → the wolf-vote buffer (R2) is **needed** (day ballot buffer reused, ~10 lines)
  - only the interrupt surfaces → the runtime buffers for us → R2's buffer is **free**, live-ship is safe
- **(b) Replay** — after `Command(resume=...)`, are the siblings' updates emitted *again*, or only the resumed branch + downstream?
  - replayed → the translator needs dedupe (skip already-assigned seqs)
  - not replayed → the translator stays dumb

The toy mirrors the production night topology exactly: Send fan-out → parallel workers (one is the "human" and interrupts) → barrier. No LLMs — every chunk must be legible.

**Record the verdicts + raw outputs below; these are runtime semantics, so the pinned version matters** (DeltaChannel upgrade may change them).

In [1]:
from importlib.metadata import version
print("langgraph", version("langgraph"))

langgraph 1.1.10


## Cell 1 — the toy graph

`fan_out` → three parallel `worker` Sends (`a`, `b` return instantly; `human` interrupts) → `barrier`.
Same shape as: `check_game_end_day` → night actors → `NIGHT_RESOLUTION`, or `wolf_fan_out_vote` → wolf votes → `collect_wolf_votes`.

In [2]:
from typing import TypedDict, Annotated
from operator import add

from langgraph.graph import StateGraph, START
from langgraph.types import interrupt, Command, Send
from langgraph.checkpoint.memory import MemorySaver


class S(TypedDict):
    log: Annotated[list[str], add]


def fan_out(state):
    return [Send("worker", {"who": w}) for w in ("a", "b", "human")]


def worker(payload):
    if payload["who"] == "human":
        answer = interrupt({"prompt": "vote?"})   # the human wolf
        return {"log": [f"human:{answer}"]}
    return {"log": [f"{payload['who']}:voted"]}    # the LLM wolves


def barrier(state):
    return {"log": ["barrier-ran"]}


g = StateGraph(S)
g.add_node("worker", worker)
g.add_node("barrier", barrier)
g.add_conditional_edges(START, fan_out, ["worker"])
g.add_edge("worker", "barrier")
graph = g.compile(checkpointer=MemorySaver())

## Cell 2 — Phase 1: run until the interrupt

**Question (a) is answered by reading this output.**
Look for `{'worker': {'log': ['a:voted']}}` / `['b:voted']` chunks **before** the `'__interrupt__'` chunk.

| you see | verdict |
|---|---|
| `a`/`b` worker chunks, then the interrupt | stream **pierces** → wolf-vote buffer NEEDED |
| only the interrupt chunk | runtime **holds siblings** → buffer free, live-ship safe |

In [17]:
cfg = {"configurable": {"thread_id": "spike-1"}}
for chunk in graph.stream({"log": []}, cfg, stream_mode="updates", version='v2'):
    print("PHASE1:", chunk)

PHASE1: {'type': 'updates', 'ns': (), 'data': {'worker': {'log': ['a:voted']}}}
PHASE1: {'type': 'updates', 'ns': (), 'data': {'worker': {'log': ['b:voted']}}}
PHASE1: {'type': 'updates', 'ns': (), 'data': {'__interrupt__': (Interrupt(value={'prompt': 'vote?'}, id='1cf8569c762b46b78b19ee06d4c0e644'),)}}


## Cell 3 — Phase 2: resume

**Question (b) is answered here.** Do `a:voted` / `b:voted` chunks appear **again** after resume?

| you see | verdict |
|---|---|
| siblings re-emitted, then human + barrier | resume **replays** → translator needs dedupe |
| only `human:v1` + `barrier-ran` | no replay → translator stays dumb |

The final-state check is a control, not a question: `log` must contain all three votes + `barrier-ran` either way — proving the *state* barrier held regardless of what the stream showed.

In [18]:
for chunk in graph.stream(Command(resume="happy"), cfg, stream_mode="updates"):
    print("PHASE2:", chunk)

print("\nFINAL STATE:", graph.get_state(cfg).values)

PHASE2: {'worker': {'log': ['a:voted']}, '__metadata__': {'cached': True}}
PHASE2: {'worker': {'log': ['b:voted']}, '__metadata__': {'cached': True}}
PHASE2: {'worker': {'log': ['human:happy']}}
PHASE2: {'barrier': {'log': ['barrier-ran']}}

FINAL STATE: {'log': ['a:voted', 'b:voted', 'human:v2', 'barrier-ran', 'a:voted', 'b:voted', 'human:happy', 'barrier-ran', 'a:voted', 'b:voted', 'human:happy', 'barrier-ran', 'a:voted', 'b:voted', 'human:happy', 'barrier-ran', 'a:voted', 'b:voted', 'human:happy', 'barrier-ran']}


In [13]:
for chunk in graph.stream(Command(resume="happy"), cfg, stream_mode="updates",version='v2'):
    print("PHASE2:", chunk)

print("\nFINAL STATE:", graph.get_state(cfg).values)

PHASE2: {'type': 'updates', 'ns': (), 'data': {'worker': {'log': ['a:voted']}, '__metadata__': {'cached': True}}}
PHASE2: {'type': 'updates', 'ns': (), 'data': {'worker': {'log': ['b:voted']}, '__metadata__': {'cached': True}}}
PHASE2: {'type': 'updates', 'ns': (), 'data': {'worker': {'log': ['human:happy']}}}
PHASE2: {'type': 'updates', 'ns': (), 'data': {'barrier': {'log': ['barrier-ran']}}}

FINAL STATE: {'log': ['a:voted', 'b:voted', 'human:v2', 'barrier-ran', 'a:voted', 'b:voted', 'human:happy', 'barrier-ran', 'a:voted', 'b:voted', 'human:happy', 'barrier-ran']}


## Verdicts (fill in)

- langgraph version: `____`
- **(a) siblings surface before resume:** YES / NO → wolf-vote buffer: NEEDED / FREE
- **(b) sibling updates replayed on resume:** YES / NO → translator dedupe: NEEDED / NOT NEEDED
- control: final `log` contained all four entries: YES / NO

Hand these two verdicts back and the translator build starts with the buffer scope settled.

## Cell 4 (optional) — the exact production topology: interrupt inside a *subgraph* inside a parallel branch

Only run this if the flat answers surprise you or you want belt-and-braces: the human **wolf** interrupts inside the compiled wolf subgraph, which is itself one branch of the parent's parallel night. Nesting *usually* doesn't change stream semantics — but this is the configuration production actually runs, so a confirming run is cheap insurance. Note `subgraphs=True`: without it the subgraph is one opaque chunk.

In [ ]:
class P(TypedDict):
    log: Annotated[list[str], add]


def sub_actor(state):
    answer = interrupt({"prompt": "sub vote?"})
    return {"log": [f"sub-human:{answer}"]}


sub = StateGraph(P)
sub.add_node("sub_actor", sub_actor)
sub.add_edge(START, "sub_actor")
sub_compiled = sub.compile()   # no checkpointer: inherits the parent's


def llm_branch(state):
    return {"log": ["llm-branch:done"]}


parent = StateGraph(P)
parent.add_node("llm_branch", llm_branch)
parent.add_node("sub_branch", sub_compiled)
parent.add_node("barrier", barrier)
parent.add_edge(START, "llm_branch")
parent.add_edge(START, "sub_branch")
parent.add_edge("llm_branch", "barrier")
parent.add_edge("sub_branch", "barrier")
parent_compiled = parent.compile(checkpointer=MemorySaver())

cfg2 = {"configurable": {"thread_id": "spike-nested-1"}}
for chunk in parent_compiled.stream({"log": []}, cfg2, stream_mode="updates", subgraphs=True):
    print("N-PHASE1:", chunk)
print("---resume---")
for chunk in parent_compiled.stream(Command(resume="v2"), cfg2, stream_mode="updates", subgraphs=True):
    print("N-PHASE2:", chunk)
print("\nFINAL STATE:", parent_compiled.get_state(cfg2).values)